# SAD AR5 — Análise de dados, otimização e validação

**CT302 · Grupo VI · Escola Naval**

Notebook reprodutível alinhado com o relatório e a plataforma web. Cobre todo o pipeline analítico:

| Secção | Conteúdo |
|--------|----------|
| 0 | Setup, caminhos e dependências |
| 1 | EDA — apreensões UNODC (séries, histogramas, boxplots) |
| 2 | Pesos AHP — matriz, consistência (CR) |
| 3 | Grelha de risco multi-ameaça — estatísticas por camada |
| 4 | Otimização — set cover, MCLP, frota 24 h, sensibilidade |
| 5 | Validação — backtest temporal, baseline patrulha |
| 6 | Respostas SAD Q1–Q3 e exportação |
| 7 | Galeria de figuras do relatório |

> **Nota:** As métricas canónicas (Q1–Q3, ganho 2,13×) estão em `resultados/validacao.json`.  
> A regeneração completa (`main.py`) pode alterar valores se os dados de entrada mudarem.


## 0. Setup — instalação e imports

In [ ]:
# Descomente na primeira execução:
# %pip install -q -r ../requirements.txt jupyter matplotlib seaborn openpyxl

import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display, Markdown

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 6)
sns.set_theme(style="whitegrid")

REPO = Path.cwd()
if not (REPO / "src" / "config.py").exists():
    REPO = REPO.parent
if not (REPO / "src" / "config.py").exists():
    raise FileNotFoundError("Abra o notebook na pasta notebooks/ do repositório")

SRC = REPO / "src"
sys.path.insert(0, str(SRC))
os.chdir(SRC)

FIGDIR = REPO / "resultados" / "figuras"
DMFIG = REPO / "resultados" / "dm"
OUTDIR = REPO / "resultados"
XLSX = REPO / "dados/fontes/apreensoes_droga_PT.xlsx"
FIGDIR.mkdir(parents=True, exist_ok=True)

def show_fig(name, title=None, folder=FIGDIR):
    p = folder / name
    if p.exists():
        if title:
            display(Markdown(f"**{title}**"))
        display(Image(filename=str(p), width=900))
    else:
        print(f"(figura em falta: {p} — execute o pipeline)")

print("Repositório:", REPO)


## 1. Análise exploratória — apreensões de droga

In [ ]:
from dm.eda import gerar as eda_gerar
from dm.preproc import carregar_e_limpar

df = carregar_e_limpar(log=False)
resumo = eda_gerar()
print(json.dumps(resumo, indent=2, ensure_ascii=False))

display(df.describe(include="all").T.head(12))
print(f"\nRegistos: {len(df)} | Anos {df['ano'].min()}–{df['ano'].max()} | Marítimas: {df['maritimo'].mean():.1%}")

show_fig("eda_panorama.png", "Panorama EDA (4 painéis)", DMFIG)


## 2. Pesos AHP (Processo de Hierarquia Analítica)

In [ ]:
from dm.ahp_pesos import main as ahp_main

ahp = ahp_main()
print(f"Razão de consistência CR = {ahp['consistency_ratio']:.4f}  →  consistente: {ahp['consistente']}")
pd.DataFrame([
    {"Ameaça": k, "Peso AHP": v, "Peso adoptado": ahp["pesos_adotados"][k]}
    for k, v in ahp["pesos_ahp"].items()
])
show_fig("24_ahp_pesos.png", "Pesos AHP por ameaça")


## 3. Grelha de risco multi-ameaça

In [ ]:
from geo import gerar_procura, bases_proj, COSTA_LONLAT, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX
from risco import calcular_risco
from config import PESOS_AMEACA, LIMIAR_RISCO_OPERACIONAL

LIMIAR = LIMIAR_RISCO_OPERACIONAL
pts = gerar_procura()
bases = bases_proj()
calcular_risco(pts, str(XLSX))

r = np.array([p["risco"] for p in pts])
stats = {
    "células": len(pts),
    "risco médio": round(float(r.mean()), 4),
    "risco máx.": round(float(r.max()), 4),
    f"alto risco (≥{LIMIAR})": int((r >= LIMIAR).sum()),
    f"% alto risco": round(100 * (r >= LIMIAR).mean(), 1),
}
pd.Series(stats, name="Grelha SAD")

ameacas = ["droga", "pesca", "poluicao", "imigracao"]
rows = []
for a in ameacas:
    v = np.array([p[f"r_{a}"] for p in pts])
    rows.append({"ameaça": a, "peso": PESOS_AMEACA[a], "média": round(v.mean(),3),
                 "máx": round(v.max(),3), "p90": round(np.percentile(v,90),3)})
pd.DataFrame(rows)

show_fig("01_risco.png", "Fig. 1 — Índice de risco agregado")
show_fig("02_ameacas.png", "Fig. 2 — Campos de intensidade por ameaça")


## 4. Otimização — cobertura, MCLP e dimensionamento de frota

In [ ]:
# Executa o pipeline de otimização (gera figuras 03–08 e resultados.json)
# Demora ~10–30 s. Comente se já tiver corrido main.py recentemente.

import main as pipeline
pipeline.main()


In [ ]:
with open(OUTDIR / "resultados.json", encoding="utf-8") as f:
    res = json.load(f)

# Cenários de vento (set cover + frota simples)
rows = []
for nome, c in res["cenarios"]["A_conservador_vento"].items():
    rows.append({
        "cenário": nome,
        "raio km": c["raio_km"],
        "bases": c["set_cover"]["n_bases"],
        "frota 24h": c["frota"]["frota_total"],
        "fora alcance": c["set_cover"]["n_nao_cobrivel"],
    })
display(pd.DataFrame(rows))

# Configuração recomendada (MCLP + persistência)
b = res["cenarios"]["B_alcance_AR5"]
print(f"k recomendado: {b['k_recomendado']} bases")
print(f"Bases: {', '.join(b['bases_recomendadas'])}")
print(f"Frota costeira 24h: {b['frota_persistencia_costeira']['frota_total']} AR5")
print(f"Frota total 24h: {b['frota_persistencia_total']['frota_total']} AR5")

show_fig("03_cobertura_conservador.png", "Fig. 3 — Cobertura conservadora (R=90 km)")
show_fig("04_cobertura_alargado.png", "Fig. 4 — Configuração recomendada MCLP")
show_fig("05_tradeoff.png", "Fig. 5 — Curva trade-off MCLP")
show_fig("06_frota.png", "Fig. 6 — Frota por cenário de vento")
show_fig("07_sensibilidade.png", "Fig. 7 — Sensibilidade (sensor, revisita, disponibilidade)")
show_fig("08_frota_vs_bases.png", "Fig. 8 — Frota mínima vs. nº de bases")


In [ ]:
# Tabela resumo exportada pelo pipeline
if (OUTDIR / "resumo_cenarios.csv").exists():
    display(pd.read_csv(OUTDIR / "resumo_cenarios.csv"))


## 5. Validação científica — backtest e baseline

In [ ]:
# Carrega métricas canónicas (não sobrescreve validacao.json)
with open(OUTDIR / "validacao.json", encoding="utf-8") as f:
    val = json.load(f)

bt = val["backtest_temporal"]
bl = val["baseline_patrulha"]

pd.DataFrame([
    {"métrica": "Holdout (anos)", "valor": bt["anos_holdout"]},
    {"métrica": "N apreensões teste", "valor": bt["n_holdout"]},
    {"métrica": "Acerto limiar alto risco", "valor": f"{bt['taxa_acerto_limiar']:.1%}"},
    {"métrica": "Ganho vs aleatório (backtest)", "valor": f"{bt['ganho_relativo_limiar']}×"},
    {"métrica": "Células patrulha SAD", "valor": bl["n_celulas_patrulha"]},
    {"métrica": "Captura risco SAD", "valor": f"{bl['pct_risco_total_capturado_sad']}%"},
    {"métrica": "Ganho SAD vs aleatório", "valor": f"{bl['ganho_sad_vs_aleatorio']}×"},
    {"métrica": "Ganho SAD vs uniforme costeira", "valor": f"{bl['ganho_sad_vs_uniforme']}×"},
])

show_fig("21_backtest_temporal.png", "Fig. 21 — Backtest temporal (droga)")
show_fig("22_baseline_patrulha.png", "Fig. 22 — Baseline patrulha")
show_fig("25_sensibilidade_limiar.png", "Fig. 25 — Sensibilidade do limiar")


In [ ]:
# Regenerar figuras de validação (opcional — pode alterar validacao.json)
# import validacao; validacao.main()


## 6. Respostas SAD — Q1, Q2, Q3

In [ ]:
resp = val["resposta_objetivo"]
pd.DataFrame([
    ("Q1 — Onde patrulhar?", resp["Q1_onde"]["resposta"]),
    ("Q2 — Quantos AR5?", resp["Q2_quantos"]["resposta"]),
    ("Q3 — Onde colocar bases?", resp["Q3_bases"]["resposta"]),
], columns=["Pergunta", "Resposta"])


## 7. Galeria — todas as figuras do relatório

In [ ]:
figs = sorted(FIGDIR.glob("*.png"))
print(f"{len(figs)} figuras em {FIGDIR}")
for p in figs:
    show_fig(p.name, p.stem.replace('_', ' '))


## Notas finais

- **Intensidades EMODnet:** `cd src && python -m dm.construir_dados_reais` (requer `rasterio`).
- **Classificação marítimo/terrestre:** `python -m dm.main_dm`.
- **Plataforma web:** ver `plataforma/ARRANQUE-MACOS.md` ou `plataforma/ARRANQUE-WINDOWS.md`.
- Os JSON em `resultados/` alimentam a API em runtime.
